# DeepFilterNet df_mlx `train_dynamic`

> This notebook now defaults to streaming datastore shards from Hugging Face using `--cache-hf sealad886/dfn4_mlx_datastore`.

> It supports both Google Colab and local VS Code/Jupyter execution.

> Tip: defaults are smoke-test friendly (small batch/epoch limits) so the full notebook can run end-to-end quickly. Increase limits for real training.

In [ ]:
import importlib.util

if importlib.util.find_spec('google.colab') is not None:
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print('Not running in Colab; skipping Google Drive mount.')

In [ ]:
import os
from pathlib import Path

IN_COLAB = Path('/content').exists()
REPO_URL = 'https://github.com/sealad886/DeepFilterNet4.git'
REPO_DIR = Path('/content/DeepFilterNet') if IN_COLAB else Path.cwd().resolve()
if not (REPO_DIR / 'DeepFilterNet').exists() and (REPO_DIR.parent / 'DeepFilterNet').exists():
    REPO_DIR = REPO_DIR.parent
PROJECT_DIR = REPO_DIR / 'DeepFilterNet'
VENV_DIR = REPO_DIR / '.venv'
VENV_PY = VENV_DIR / 'bin' / 'python'

# Data source (priority order in this notebook): CACHE_HF_REPO -> CACHE_DIR -> DATASET_CONFIG_JSON
CACHE_HF_REPO = 'sealad886/dfn4_mlx_datastore'
CACHE_DIR = None  # Example: Path('/content/drive/MyDrive/DeepFilterNet/cache/mlx_datastore')
DATASET_CONFIG_JSON = Path('/content/drive/MyDrive/DeepFilterNet/data/config.json') if IN_COLAB else None

# Optional run-config TOML
RUN_CONFIG_TOML = None  # Example: Path('/content/drive/MyDrive/DeepFilterNet/configs/run_config.toml')

# Checkpoints and run shape (smoke-test defaults)
CHECKPOINT_DIR = (
    Path('/content/drive/MyDrive/DeepFilterNet/checkpoints/df_mlx_train_dynamic')
    if IN_COLAB
    else REPO_DIR / 'out' / 'checkpoints' / 'df_mlx_train_dynamic'
)
EPOCHS = 1
BATCH_SIZE = 2
BACKBONE = 'mamba'  # mamba | gru | attention
PRESET = 'debug'    # entry | pro | max | ultra | debug

# Extra CLI flags (space-separated string)
EXTRA_FLAGS = '--max-train-batches 2 --max-valid-batches 1 --validate-every 1'

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
print('Configuration ready.')
print('IN_COLAB:', IN_COLAB)
print('REPO_DIR:', REPO_DIR)
print('PROJECT_DIR:', PROJECT_DIR)
print('CHECKPOINT_DIR:', CHECKPOINT_DIR)
print('CACHE_HF_REPO:', CACHE_HF_REPO)
print('VENV_PY exists:', VENV_PY.exists())

In [ ]:
%%bash
set -euo pipefail

if [[ -d /content ]]; then
  if [[ ! -d /content/DeepFilterNet/.git ]]; then
    git clone --depth 1 https://github.com/sealad886/DeepFilterNet4.git /content/DeepFilterNet
  else
    git -C /content/DeepFilterNet fetch --all --prune
    git -C /content/DeepFilterNet pull --ff-only
  fi

  apt-get update -y
  apt-get install -y python3.10 python3.10-venv libsndfile1 ffmpeg cargo libportaudio2 python3-pyaudio libportaudio-ocaml-dev
else
  echo 'Non-Colab runtime detected; skipping clone + apt bootstrap.'
fi

In [ ]:
%%bash
set -euo pipefail

REPO_DIR="${PWD}"
if [[ ! -d "${REPO_DIR}/DeepFilterNet" && -d "${PWD}/../DeepFilterNet" ]]; then
  REPO_DIR="$(cd "${PWD}/.." && pwd)"
fi
if [[ -d /content/DeepFilterNet ]]; then
  REPO_DIR="/content/DeepFilterNet"
fi

VENV_DIR="${REPO_DIR}/.venv"
VENV_PY="${VENV_DIR}/bin/python"

echo "REPO_DIR: ${REPO_DIR}"
echo "VENV_DIR: ${VENV_DIR}"
echo "VENV_PY: ${VENV_PY}"

if [[ -d /content ]]; then
  echo "$ python3.10 -m venv ${VENV_DIR}"
  python3.10 -m venv "${VENV_DIR}"

  echo "$ ${VENV_PY} -m pip install --upgrade pip setuptools wheel"
  "${VENV_PY}" -m pip install --upgrade pip setuptools wheel

  echo "$ bash setup.sh --all --venv ${VENV_DIR}"
  (cd "${REPO_DIR}" && bash setup.sh --all --venv "${VENV_DIR}")

  MLX_PKG="mlx[cpu]"
  if command -v nvidia-smi >/dev/null 2>&1; then
    MLX_PKG="mlx[cuda12]"
  fi
  echo "$ ${VENV_PY} -m pip install --upgrade ${MLX_PKG}"
  "${VENV_PY}" -m pip install --upgrade "${MLX_PKG}"
else
  echo 'Non-Colab runtime detected; using existing local .venv and skipping setup.sh/bootstrap.'
fi

if [[ ! -x "${VENV_PY}" ]]; then
  echo "Missing Python interpreter at ${VENV_PY}"
  exit 1
fi

echo "$ ${VENV_PY} -c 'import mlx, df_mlx.train_dynamic; print(\"mlx+df_mlx import OK\")'"
(cd "${REPO_DIR}/DeepFilterNet" && "${VENV_PY}" -c 'import mlx, df_mlx.train_dynamic; print("mlx+df_mlx import OK")')

In [ ]:
import os
import shlex
import subprocess
from pathlib import Path

if not VENV_PY.exists():
    raise FileNotFoundError(f'Python interpreter not found: {VENV_PY}')
if not PROJECT_DIR.exists():
    raise FileNotFoundError(f'Project dir not found: {PROJECT_DIR}')

cache_hf_repo = (CACHE_HF_REPO or '').strip()
if cache_hf_repo.startswith('hf://'):
    cache_hf_repo = cache_hf_repo.removeprefix('hf://')
if cache_hf_repo.startswith('datasets/'):
    cache_hf_repo = cache_hf_repo.removeprefix('datasets/')

if not cache_hf_repo and CACHE_DIR is None:
    if DATASET_CONFIG_JSON is None or not Path(DATASET_CONFIG_JSON).exists():
        raise FileNotFoundError(
            'Provide CACHE_HF_REPO or CACHE_DIR, or set DATASET_CONFIG_JSON to an existing config file.'
        )

cmd = [
    str(VENV_PY),
    '-m',
    'df_mlx.train_dynamic',
    '--checkpoint-dir',
    str(CHECKPOINT_DIR),
    '--epochs',
    str(EPOCHS),
    '--batch-size',
    str(BATCH_SIZE),
    '--backbone-type',
    BACKBONE,
    '--preset',
    PRESET,
]

if cache_hf_repo:
    cmd.extend(['--cache-hf', cache_hf_repo])
elif CACHE_DIR is not None:
    cmd.extend(['--cache-dir', str(CACHE_DIR)])
else:
    cmd.extend(['--config', str(DATASET_CONFIG_JSON)])

if RUN_CONFIG_TOML is not None:
    run_cfg = Path(RUN_CONFIG_TOML)
    if not run_cfg.exists():
        raise FileNotFoundError(f'Run config not found: {run_cfg}')
    cmd.extend(['--run-config', str(run_cfg)])

if EXTRA_FLAGS.strip():
    cmd.extend(shlex.split(EXTRA_FLAGS))

print('Training command:')
print(' '.join(shlex.quote(part) for part in cmd))

env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'
env['DFNET_TQDM'] = '1'

print('Starting training...')
subprocess.run(cmd, cwd=str(PROJECT_DIR), env=env, check=True)

ckpt_path = Path(CHECKPOINT_DIR)
if not ckpt_path.exists():
    print('No checkpoint directory found yet.')
else:
    print('Recent checkpoint artifacts:')
    files = sorted(ckpt_path.glob('*'), key=lambda p: p.stat().st_mtime, reverse=True)
    for p in files[:20]:
        print(p.name)